[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/information_theory/06_information_theory_in_deep_learning/first_principles.ipynb)

# Topic 06: Information Theory in Deep Learning

## 1. First-Principles Intuition & Motivation

Every objective in this notebook answers one of two questions.

1. **How many bits does this model need to describe the data?** — cross-entropy, negative log-likelihood, description length, bits per byte.
2. **How many bits does this representation keep, and about what?** — mutual information, rate, the KL term of a VAE, the InfoNCE score.

Topics 01–05 built the vocabulary. This module shows that the objectives practitioners actually type into a training script are these two questions traded off against each other.

### The Universal Move: Bound What You Cannot Compute

Every quantity we want is intractable. The marginal likelihood $\log p(x) = \log\int p(x, z)\,dz$ integrates over all latents; the mutual information $I(X; Z)$ needs the marginal $q(z)$; the posterior $p(z \mid x)$ is unavailable.

The remedy is always the same three-step move:

1. Introduce an auxiliary distribution (a variational posterior $q_\phi(z \mid x)$, a critic $f_\theta$, a decoder $q(x \mid z)$).
2. Show that the resulting expression is a bound on the target.
3. Show that the **gap is a KL divergence**, so tightening the bound is itself a learning problem with a known optimum.

Because the gap is a KL, we always know exactly what "tight" means and what the optimal auxiliary object is.

### The Universal Trade-Off: Rate versus Distortion

Compression theory gives the shape of every representation-learning objective:

$$
\min_{\text{encoder}} \; \underbrace{D}_{\text{distortion: how badly we reconstruct or predict}} \; + \; \lambda \underbrace{R}_{\text{rate: how many bits the code carries}}
$$

| Objective | Distortion term | Rate term | Multiplier |
|---|---|---|---|
| VAE (ELBO) | $\mathbb{E}_q\left[-\log p(x \mid z)\right]$ | $\mathbb{E}_x D_{\mathrm{KL}}\left(q(z \mid x) \parallel p(z)\right)$ | 1 |
| $\beta$-VAE | same | same | $\beta$ |
| Information Bottleneck | $-I(Z; Y)$ | $I(X; Z)$ | $1/\beta$ |
| Neural image codec | perceptual/MSE loss | entropy-model bitrate | $\lambda$ |
| RLHF | $-\mathbb{E}\left[r(x, y)\right]$ | $D_{\mathrm{KL}}\left(\pi \parallel \pi_{\text{ref}}\right)$ | $\beta$ |

One picture, five literatures. The multiplier is always an exchange rate in "units of distortion per nat".

### Compression Is Learning

Given a probabilistic model $q$, arithmetic coding compresses a dataset $\mathcal{D}$ to $-\log_2 q(\mathcal{D})$ bits plus negligible overhead. So a model's log-likelihood *is* a compression rate, and "the better model" and "the better compressor" are the same claim.

This is why language-model progress is reported in bits per byte, why the irreducible constant in scaling laws estimates the entropy rate of text, and why the minimum description length principle can convert a compression bound into a generalization bound. Section 3 makes the last step precise via bits-back coding.

## 2. Rigorous Mathematical Definitions & Theorem Statements

### Definition 2.1 (Evidence Lower Bound)

For a latent-variable model $p_\theta(x, z) = p_\theta(x \mid z)p(z)$ and any variational family $q_\phi(z \mid x)$,

$$
\mathcal{L}_{\mathrm{ELBO}}(\theta, \phi; x) = \mathbb{E}_{q_\phi(z \mid x)}\left[\log p_\theta(x \mid z)\right] - D_{\mathrm{KL}}\left(q_\phi(z \mid x) \parallel p(z)\right)
$$

The $\beta$-weighted version multiplies the second term by $\beta \gt 0$.

### Definition 2.2 (Rate and Distortion of a Stochastic Encoder)

With data distribution $p(x)$ and encoder $q(z \mid x)$, define the aggregate posterior $q(z) = \mathbb{E}_{p(x)}\left[q(z \mid x)\right]$ and

$$
R = \mathbb{E}_{p(x)}\left[D_{\mathrm{KL}}\left(q(z \mid x) \parallel p(z)\right)\right], \qquad D = \mathbb{E}_{p(x)}\mathbb{E}_{q(z \mid x)}\left[-\log p(x \mid z)\right]
$$

$R$ is the average number of nats the code spends per example under prior $p(z)$; $D$ is the reconstruction cost in nats.

### Definition 2.3 (Information Bottleneck)

Given $p(x, y)$ and a stochastic encoder $p(z \mid x)$ forming the Markov chain $Y \to X \to Z$, the IB objective is

$$
\mathcal{L}_{\mathrm{IB}} = I(X; Z) - \beta\, I(Z; Y)
$$

to be **minimized** over $p(z \mid x)$. The **information curve** is the set of achievable pairs $\left(I(X; Z), I(Z; Y)\right)$ on the upper boundary; $\beta$ traces it.

### Definition 2.4 (Description Length and the MDL Principle)

A two-part code describes a hypothesis $M$ in $L(M)$ bits and then the data given the hypothesis in $L(\mathcal{D} \mid M) = -\log_2 p(\mathcal{D} \mid M)$ bits. The MDL principle selects

$$
\hat{M} = \arg\min_{M}\left\{L(M) + L(\mathcal{D} \mid M)\right\}
$$

The one-part (Bayesian/normalized-maximum-likelihood) refinement replaces the sum by $-\log_2 p(\mathcal{D}) = -\log_2 \int p(\mathcal{D} \mid M)\pi(M)\,dM$.

### Theorem Statements

- **Theorem A (ELBO decomposition)**: $\log p_\theta(x) = \mathcal{L}_{\mathrm{ELBO}}(\theta, \phi; x) + D_{\mathrm{KL}}\left(q_\phi(z \mid x) \parallel p_\theta(z \mid x)\right)$; hence the ELBO is a lower bound, tight iff $q_\phi$ equals the true posterior.
- **Theorem B (Rate contains mutual information)**: $R = I(X; Z) + D_{\mathrm{KL}}\left(q(z) \parallel p(z)\right) \ge I(X; Z)$, with equality iff the aggregate posterior matches the prior.
- **Theorem C (Bounds on the ELBO terms)**: $H - D \le I(X; Z) \le R$ where $H$ is the data entropy; the achievable region in the $(R, D)$ plane lies above $D \ge H - R$.
- **Theorem D (IB self-consistent equations)**: at a stationary point of $\mathcal{L}_{\mathrm{IB}}$, $p(z \mid x) = \frac{p(z)}{Z(x, \beta)}\exp\left(-\beta\, D_{\mathrm{KL}}\left(p(y \mid x) \parallel p(y \mid z)\right)\right)$, together with $p(z) = \sum_x p(x)p(z \mid x)$ and $p(y \mid z) = \sum_x p(y \mid x)p(x \mid z)$.
- **Theorem E (IB curve)**: $I(Z; Y)$ as a function of $I(X; Z)$ is concave and nondecreasing, and its slope at the optimum is $1/\beta$.
- **Theorem F (KL-regularized optimum)**: $\arg\max_{\pi}\left\{\mathbb{E}_{\pi}\left[r(y)\right] - \beta D_{\mathrm{KL}}\left(\pi \parallel \pi_{\text{ref}}\right)\right\} = \pi^{*}(y) \propto \pi_{\text{ref}}(y)e^{r(y)/\beta}$, with optimal value $\beta\log \mathbb{E}_{\pi_{\text{ref}}}\left[e^{r/\beta}\right]$.
- **Theorem G (InfoNCE optimal critic)**: the minimizer of the contrastive loss is $f^{*}(x, y) = \log\frac{p(y \mid x)}{p(y)} + c(x)$, and $I(X; Y) \ge \log K - \mathcal{L}_{\mathrm{NCE}}$.
- **Theorem H (Bits-back)**: a sender who shares the prior $p(z)$ with the receiver can transmit $x$ using $\mathbb{E}_q\left[-\log p(x \mid z)\right] + D_{\mathrm{KL}}\left(q(z \mid x) \parallel p(z)\right)$ nats on average — exactly the negative ELBO.

## 3. Step-by-Step Mathematical Proofs & Derivations

### Proof 3.1 (The ELBO and its gap)

**Claim.** $\log p_\theta(x) = \mathcal{L}_{\mathrm{ELBO}} + D_{\mathrm{KL}}\left(q_\phi(z \mid x) \parallel p_\theta(z \mid x)\right)$.

**Step 1 (multiply and divide).** For any $q_\phi(z \mid x)$ with support containing that of the posterior,

$$
\log p_\theta(x) = \log\int p_\theta(x, z)\,dz = \log\int q_\phi(z \mid x)\,\frac{p_\theta(x, z)}{q_\phi(z \mid x)}\,dz
$$

**Step 2 (Jensen).** The logarithm is concave, so

$$
\log p_\theta(x) \ge \mathbb{E}_{q_\phi}\left[\log\frac{p_\theta(x, z)}{q_\phi(z \mid x)}\right] = \mathbb{E}_{q_\phi}\left[\log p_\theta(x \mid z)\right] - D_{\mathrm{KL}}\left(q_\phi(z \mid x) \parallel p(z)\right)
$$

using $p_\theta(x, z) = p_\theta(x \mid z)p(z)$.

**Step 3 (exact gap, no inequality needed).** Write $p_\theta(x, z) = p_\theta(z \mid x)p_\theta(x)$ instead:

$$
\mathbb{E}_{q_\phi}\left[\log\frac{p_\theta(x, z)}{q_\phi(z \mid x)}\right] = \log p_\theta(x) - \mathbb{E}_{q_\phi}\left[\log\frac{q_\phi(z \mid x)}{p_\theta(z \mid x)}\right] = \log p_\theta(x) - D_{\mathrm{KL}}\left(q_\phi \parallel p_\theta(\cdot \mid x)\right)
$$

Rearranging gives the identity, and nonnegativity of KL re-derives Step 2.

$$
\boxed{\log p_\theta(x) = \mathcal{L}_{\mathrm{ELBO}} + D_{\mathrm{KL}}\left(q_\phi(z \mid x) \parallel p_\theta(z \mid x)\right) \ge \mathcal{L}_{\mathrm{ELBO}}}
$$

**Interpretation.** Maximizing the ELBO in $\theta$ fits the data; maximizing it in $\phi$ closes the inference gap. The gap is a *reverse* KL, which is why variational posteriors are mode-seeking and typically underdispersed. $\blacksquare$

### Proof 3.2 (The rate contains the mutual information)

**Claim.** $R = \mathbb{E}_{p(x)}\left[D_{\mathrm{KL}}\left(q(z \mid x) \parallel p(z)\right)\right] = I(X; Z) + D_{\mathrm{KL}}\left(q(z) \parallel p(z)\right)$.

**Step 1 (insert the aggregate posterior).** With $q(z) = \mathbb{E}_{p(x)}\left[q(z \mid x)\right]$, split the log-ratio:

$$
\log\frac{q(z \mid x)}{p(z)} = \log\frac{q(z \mid x)}{q(z)} + \log\frac{q(z)}{p(z)}
$$

**Step 2 (take expectations under $p(x)q(z \mid x)$).** The first term averages to

$$
\mathbb{E}_{p(x)q(z \mid x)}\left[\log\frac{q(z \mid x)}{q(z)}\right] = I(X; Z)
$$

by the definition of mutual information for the joint $p(x)q(z \mid x)$. The second term depends on $z$ only, and $z$ has marginal $q(z)$, so it averages to $D_{\mathrm{KL}}\left(q(z) \parallel p(z)\right)$.

**Step 3 (combine).**

$$
R = I(X; Z) + D_{\mathrm{KL}}\left(q(z) \parallel p(z)\right) \ge I(X; Z)
$$

with equality iff the aggregate posterior matches the prior.

$$
\boxed{R = I(X; Z) + D_{\mathrm{KL}}\left(q(z) \parallel p(z)\right) \ge I(X; Z)}
$$

**Consequences.** (i) The VAE's KL term over-charges for information whenever $q(z) \neq p(z)$ — the "prior hole" problem, addressed by learned or flow-based priors. (ii) Since $R \ge I(X; Z)$ and $R = 0$ forces $I(X; Z) = 0$, driving the KL term to zero is *exactly* posterior collapse: the latent becomes independent of the data. $\blacksquare$

### Proof 3.3 (Rate–distortion bounds on the ELBO: the "broken ELBO" picture)

**Setup.** Let $H = -\mathbb{E}_{p(x)}\left[\log p(x)\right]$ be the data entropy (a constant of the dataset), $R$ and $D$ as in Definition 2.2.

**Step 1 (upper bound on $I(X; Z)$).** Proof 3.2 gives $I(X; Z) \le R$ directly.

**Step 2 (lower bound on $I(X; Z)$).** Write $I(X; Z) = H(X) - H(X \mid Z)$ and bound the conditional entropy by a variational decoder $p(x \mid z)$ (Barber–Agakov):

$$
H(X \mid Z) \le \mathbb{E}_{p(x)q(z \mid x)}\left[-\log p(x \mid z)\right] = D
$$

because the slack is $\mathbb{E}\left[D_{\mathrm{KL}}\left(q(x \mid z) \parallel p(x \mid z)\right)\right] \ge 0$. Hence

$$
I(X; Z) \ge H - D
$$

**Step 3 (the feasible region).** Combining,

$$
H - D \le I(X; Z) \le R \quad \Longrightarrow \quad D \ge H - R
$$

so no autoencoder can sit below the line $D + R = H$ in the rate–distortion plane. Every point on that line has the *same* ELBO value $-(D + R) = -H$.

**Step 4 (why this matters).** A powerful decoder can achieve $D \approx H$ with $R \approx 0$ (ignoring the latent entirely) or $D \approx 0$ with $R \approx H$ (a lossless code), and both attain the optimal ELBO. The ELBO alone therefore cannot select a representation — only the multiplier $\beta$ (or an explicit rate target) chooses a point on the line.

$$
\boxed{D \ge H - R; \quad \text{the ELBO is constant along } D + R = H}
$$

**Interpretation.** This is the formal explanation of posterior collapse in VAEs with autoregressive decoders: the model slides along an isoline of the objective to the zero-rate end. $\blacksquare$

### Proof 3.4 (Information Bottleneck: the self-consistent equations)

**Setup.** Minimize $\mathcal{L} = I(X; Z) - \beta I(Z; Y)$ over the encoder $p(z \mid x)$, subject to normalization $\sum_z p(z \mid x) = 1$ for each $x$, with the Markov constraint $Y \to X \to Z$ (so $p(y \mid z) = \sum_x p(y \mid x)p(x \mid z)$).

**Step 1 (write the Lagrangian).**

$$
\mathcal{F} = I(X; Z) - \beta I(Z; Y) + \sum_x \lambda(x)\sum_z p(z \mid x)
$$

**Step 2 (differentiate the rate term).** Work in nats and use $I(X; Z) = \sum_{x,z} p(x)p(z \mid x)\ln\frac{p(z \mid x)}{p(z)}$. The dependence through $p(z) = \sum_{x'} p(x')p(z \mid x')$ contributes $-\sum_z p(x)\frac{p(z \mid x)}{p(z)}\cdot p(z)$-type terms that cancel against the explicit $+1$ by normalization, leaving

$$
\frac{\partial I(X; Z)}{\partial p(z \mid x)} = p(x)\left[\ln\frac{p(z \mid x)}{p(z)} + 1\right]
$$

**Step 3 (differentiate the relevance term).** Similarly, using the Markov structure,

$$
\frac{\partial I(Z; Y)}{\partial p(z \mid x)} = p(x)\left[\sum_y p(y \mid x)\log\frac{p(y \mid z)}{p(y)}\right] + (\text{terms vanishing by normalization})
$$

**Step 4 (set the derivative to zero).**

$$
\log\frac{p(z \mid x)}{p(z)} = \beta\sum_y p(y \mid x)\log\frac{p(y \mid z)}{p(y)} - \tilde\lambda(x)
$$

Recognize the sum: $\sum_y p(y \mid x)\log\frac{p(y \mid z)}{p(y)} = -D_{\mathrm{KL}}\left(p(y \mid x) \parallel p(y \mid z)\right) + \left[\text{terms depending only on } x\right]$, which can be absorbed into $\tilde\lambda(x)$.

**Step 5 (exponentiate).**

$$
p(z \mid x) = \frac{p(z)}{Z(x, \beta)}\exp\left(-\beta\, D_{\mathrm{KL}}\left(p(y \mid x) \parallel p(y \mid z)\right)\right)
$$

with $Z(x, \beta)$ the normalizer. Together with $p(z) = \sum_x p(x)p(z \mid x)$ and $p(y \mid z) = \frac{1}{p(z)}\sum_x p(y \mid x)p(z \mid x)p(x)$, these three coupled equations are solved by alternating updates — the **Blahut–Arimoto-style IB algorithm**.

$$
\boxed{p(z \mid x) \propto p(z)\exp\left(-\beta\, D_{\mathrm{KL}}\left(p(y \mid x) \parallel p(y \mid z)\right)\right)}
$$

**Interpretation.** The optimal encoder is a *soft clustering*: $x$ is assigned to cluster $z$ in proportion to how well the cluster's predictive distribution $p(y \mid z)$ matches $x$'s own $p(y \mid x)$, with $\beta$ as an inverse temperature. At $\beta \to 0$ everything collapses to one cluster ($I(X; Z) = 0$); as $\beta$ grows, clusters split at a sequence of phase transitions. $\blacksquare$

### Proof 3.5 (KL-regularized RL: the tilted optimal policy)

**Claim.** For fixed $x$, $\pi^{*} = \arg\max_{\pi}\left\{\mathbb{E}_{\pi}\left[r(y)\right] - \beta D_{\mathrm{KL}}\left(\pi \parallel \pi_{\text{ref}}\right)\right\}$ satisfies $\pi^{*}(y) = \frac{1}{Z}\pi_{\text{ref}}(y)e^{r(y)/\beta}$.

**Step 1 (rewrite the objective as a single KL).** Define the tilted distribution $\pi^{*}(y) = \frac{\pi_{\text{ref}}(y)e^{r(y)/\beta}}{Z}$ with $Z = \mathbb{E}_{\pi_{\text{ref}}}\left[e^{r/\beta}\right]$. Then for any $\pi$,

$$
D_{\mathrm{KL}}\left(\pi \parallel \pi^{*}\right) = \mathbb{E}_{\pi}\left[\log\frac{\pi(y)}{\pi_{\text{ref}}(y)}\right] - \frac{1}{\beta}\mathbb{E}_{\pi}\left[r(y)\right] + \log Z
$$

**Step 2 (multiply by $-\beta$).**

$$
-\beta\,D_{\mathrm{KL}}\left(\pi \parallel \pi^{*}\right) = \mathbb{E}_{\pi}\left[r(y)\right] - \beta D_{\mathrm{KL}}\left(\pi \parallel \pi_{\text{ref}}\right) - \beta\log Z
$$

so the objective equals $\beta\log Z - \beta D_{\mathrm{KL}}\left(\pi \parallel \pi^{*}\right)$.

**Step 3 (maximize).** The first term does not depend on $\pi$ and the second is nonpositive, maximized (at zero) exactly when $\pi = \pi^{*}$.

$$
\boxed{\pi^{*}(y) \propto \pi_{\text{ref}}(y)\,e^{r(y)/\beta}, \qquad \max_{\pi}\left\{\cdot\right\} = \beta\log\mathbb{E}_{\pi_{\text{ref}}}\left[e^{r/\beta}\right]}
$$

**Interpretation.**

- The optimum is an exponential *tilt* of the reference model, never a distribution outside its support — the mathematical reason a KL penalty prevents mode collapse onto reward-hacking text.
- The optimal value $\beta\log Z$ is a **free energy**; $\beta$ is a temperature, and $\beta \to 0$ recovers greedy reward maximization while $\beta \to \infty$ returns the reference model.
- Inverting the relation gives $r(y) = \beta\log\frac{\pi^{*}(y)}{\pi_{\text{ref}}(y)} + \beta\log Z$, which is precisely the reparameterization that turns RLHF into the supervised **DPO** objective. $\blacksquare$

### Proof 3.6 (The optimal InfoNCE critic is the log density ratio)

**Setup.** Given $x$ and a set $\{y_1, \dots, y_K\}$ containing exactly one true partner drawn from $p(y \mid x)$ and $K - 1$ drawn from $p(y)$, the contrastive task is to identify the positive index.

**Step 1 (posterior over the index).** By Bayes' rule, the probability that index $j$ is the positive is proportional to the likelihood of that configuration:

$$
\Pr\left[j\right] \propto p(y_j \mid x)\prod_{k \neq j} p(y_k) = \frac{p(y_j \mid x)}{p(y_j)}\prod_{k} p(y_k)
$$

The product is common to all $j$, so

$$
\Pr\left[j \mid x, y_{1:K}\right] = \frac{r(x, y_j)}{\sum_{k=1}^{K} r(x, y_k)}, \qquad r(x, y) = \frac{p(y \mid x)}{p(y)}
$$

**Step 2 (match the softmax form).** The InfoNCE model assigns $\frac{e^{f(x, y_j)}}{\sum_k e^{f(x, y_k)}}$. Cross-entropy is a strictly proper scoring rule (Topic 03), so it is minimized when the model equals the true posterior, i.e., when

$$
e^{f(x, y)} \propto r(x, y) \quad \Longleftrightarrow \quad f^{*}(x, y) = \log\frac{p(y \mid x)}{p(y)} + c(x)
$$

The additive $c(x)$ is invisible to the softmax.

**Step 3 (consequences).** The optimal critic is the *pointwise mutual information*. Therefore a trained contrastive model's logits estimate PMI up to a per-anchor constant, which is why cosine-similarity retrieval scores from CLIP-style models behave like calibrated relevance ratios, and why a learned temperature $\tau$ (dividing the logits) rescales the implied ratio.

$$
\boxed{f^{*}(x, y) = \log\frac{p(y \mid x)}{p(y)} + c(x) = \mathrm{PMI}(x, y) + c(x)}
$$

Substituting $f^{*}$ into the loss and applying Jensen yields the bound $I(X; Y) \ge \log K - \mathcal{L}_{\mathrm{NCE}}$ derived in Topic 05. $\blacksquare$

### Proof 3.7 (Bits-back coding: the negative ELBO is an achievable code length)

**Claim.** A sender and receiver who share $p(z)$ and $p(x \mid z)$ can transmit $x$ at an average cost of $-\mathcal{L}_{\mathrm{ELBO}}$ nats.

**Step 1 (the naive two-part cost).** Sample $z \sim q(z \mid x)$, transmit $z$ under the shared prior at cost $-\log p(z)$ nats, then transmit $x$ under $p(x \mid z)$ at cost $-\log p(x \mid z)$. Total: $\mathbb{E}_q\left[-\log p(z) - \log p(x \mid z)\right]$.

**Step 2 (the refund).** The sender did not need fresh randomness to choose $z$: it can *decode* $z$ from a stream of previously-queued bits using $q(z \mid x)$ as the code (this is the "bits-back" trick, realized concretely by asymmetric numeral systems in modern codecs). Those bits are recovered by the receiver, who — after reconstructing $x$ — can re-run $q(z \mid x)$ and re-encode $z$, extracting $-\log q(z \mid x)$ nats of *useful message* back.

**Step 3 (net cost).**

$$
\mathbb{E}_q\left[-\log p(z) - \log p(x \mid z) + \log q(z \mid x)\right] = \mathbb{E}_q\left[-\log p(x \mid z)\right] + D_{\mathrm{KL}}\left(q(z \mid x) \parallel p(z)\right)
$$

which is exactly $-\mathcal{L}_{\mathrm{ELBO}}(x)$.

**Step 4 (optimality).** Since $-\mathcal{L}_{\mathrm{ELBO}} \ge -\log p(x)$ with the gap equal to the posterior KL (Proof 3.1), the scheme is optimal iff the variational posterior is exact — and its inefficiency is precisely the inference gap.

$$
\boxed{\text{code length} = \mathbb{E}_q\left[-\log p(x \mid z)\right] + D_{\mathrm{KL}}\left(q(z \mid x) \parallel p(z)\right) = -\mathcal{L}_{\mathrm{ELBO}}}
$$

**Interpretation.** The KL "regularizer" is not a heuristic penalty: it is a real number of bits that a real compressor really spends. Applied to *weights* rather than latents (Hinton & van Camp, 1993), the same argument makes the variational free energy of a Bayesian neural network a description length, giving MDL-flavored generalization bounds. $\blacksquare$

## 4. Computational & Algorithmic Insights

### Implementing Variational Objectives

- **Reparameterization**: sample $z = \mu_\phi(x) + \sigma_\phi(x)\odot\epsilon$ with $\epsilon \sim \mathcal{N}(0, I)$ so gradients flow through the sampler; the score-function (REINFORCE) alternative has far higher variance and is reserved for discrete latents (with Gumbel-softmax or control variates).
- **Analytic KL**: for Gaussian posterior and standard-normal prior use the closed form $\tfrac{1}{2}\sum_j\left(\mu_j^2 + \sigma_j^2 - 1 - \log\sigma_j^2\right)$; never estimate it by sampling when a closed form exists.
- **Free bits / KL thresholding**: clamp each dimension's KL at a floor $\lambda$, i.e., use $\sum_j \max(\lambda, D_j)$, to stop the optimizer from zeroing out dimensions early.
- **KL annealing**: warm up the multiplier from 0 to $\beta$ over the first epochs so the decoder becomes useful before the rate is charged for.
- **Monitor $R$ and $D$ separately.** Reporting only the ELBO hides the position on the $D + R = H$ line (Proof 3.3) — the single most common diagnostic failure in VAE work.

### Implementing Contrastive and MI Objectives

- **In-batch negatives** make $K$ equal to the batch size; memory banks and momentum encoders (MoCo) decouple $K$ from the gradient batch to raise the $\log K$ ceiling cheaply.
- **Temperature $\tau$**: logits are $f(x, y)/\tau$; small $\tau$ sharpens the softmax and emphasizes hard negatives, effectively rescaling the implied PMI of Proof 3.6.
- **Symmetrized loss**: CLIP averages the image-to-text and text-to-image cross-entropies, bounding $I$ from both conditionals.
- **Numerical care**: contrastive losses are `log_softmax` over a $K \times K$ similarity matrix — use the fused form, mask the diagonal correctly, and normalize embeddings before the dot product.

### KL Control in RLHF and PPO

- **Which direction?** RLHF uses the *reverse* KL $D_{\mathrm{KL}}\left(\pi_\theta \parallel \pi_{\text{ref}}\right)$, evaluated on samples from $\pi_\theta$ — mode-seeking, keeping the policy inside the reference's support.
- **Estimator**: the $k_3 = (r - 1) - \log r$ estimator (unbiased, nonnegative, low variance) is standard; $k_1$ is unbiased but signed and noisy.
- **Fixed penalty vs adaptive controller**: a fixed $\beta$ fixes the exchange rate but lets the realized KL drift; an adaptive controller targets a KL budget (e.g., 10 nats) by adjusting $\beta$ — the practical form of the constrained/Lagrangian duality.
- **Reading the budget**: a KL of $k$ nats against the reference means the tuned policy would need $k$ extra nats per response to be coded under the reference model — a directly interpretable measure of how far fine-tuning moved the distribution.

## 5. Real-World Physics & AI/ML Applications

### Language Models as Compressors

Reporting a language model in **bits per byte** ($\mathcal{L}/\ln 2$ divided by bytes per token) makes the compression claim explicit and comparable across tokenizers. Large models are, empirically, state-of-the-art general-purpose lossless compressors when paired with arithmetic coding — including on modalities they were not trained on.

Scaling laws fit $L(N, D) = \frac{A}{N^{\alpha}} + \frac{B}{D^{\beta}} + L_{\infty}$, where the constant $L_{\infty}$ is an estimate of the entropy rate of the data distribution. Progress is therefore measured as *bits saved above an irreducible floor*, exactly the decomposition $H(p, q) = H(p) + D_{\mathrm{KL}}(p \parallel q)$ from Topic 03: scaling reduces the KL term, and nothing reduces the entropy term.

### Representation Learning, Bottlenecks, and the Controversy

- **Deep Variational Information Bottleneck (VIB)** implements the IB Lagrangian with variational bounds: an upper bound $R$ on $I(X; Z)$ (Proof 3.2) and a Barber–Agakov lower bound on $I(Z; Y)$. It improves adversarial robustness and calibration by explicitly capping the rate.
- **InfoGAN** adds a mutual-information regularizer $I(c; G(z, c))$ between a latent code and the generated sample, again via a variational lower bound; the result is axis-aligned, interpretable latent factors.
- **The information-plane controversy**: Shwartz-Ziv and Tishby reported a two-phase "fitting then compression" dynamic in $\left(I(X; Z), I(Z; Y)\right)$ trajectories. Saxe et al. showed the compression phase depends on saturating nonlinearities and on the binning estimator; with ReLU networks it often disappears, and for deterministic encoders with continuous inputs $I(X; Z)$ is infinite. The DPI conclusions of Topic 05 remain valid; the dynamical claims do not follow from them.

### Alignment, Compression, and Physics

- **RLHF and DPO**: Proof 3.5 shows the KL-penalized optimum is a tilted reference model and that reward can be re-expressed as a log policy ratio — the algebraic identity underlying direct preference optimization.
- **Neural compression**: learned image and video codecs minimize $D + \lambda R$ with $R$ the cross-entropy under a learned entropy model; the same Lagrangian, with a real bitstream at the end.
- **MDL and generalization**: PAC-Bayes bounds have the form $\text{test risk} \le \text{train risk} + \sqrt{\frac{D_{\mathrm{KL}}(Q \parallel P) + \log(n/\delta)}{2n}}$, where the KL is the description length of the learned posterior over weights relative to a prior — compression of the *hypothesis*, not the data, controls generalization.
- **Physics parallel**: the objective $\mathbb{E}_{\pi}[r] - \beta D_{\mathrm{KL}}(\pi \parallel \pi_{\text{ref}})$ is a variational free energy with $\beta$ as temperature and $\beta\log Z$ as the free energy itself; simulated annealing, Boltzmann policies, and diffusion samplers are all instances of the same variational principle.

## 6. Canonical Literature Mapping & References

| Concept in this notebook | Canonical source |
|---|---|
| ELBO, reparameterization, Gaussian KL | Kingma & Welling (2014), *Auto-Encoding Variational Bayes* |
| Rate–distortion view of the ELBO, $D + R \ge H$ | Alemi et al. (2018), *Fixing a Broken ELBO* |
| $\beta$-VAE and disentanglement | Higgins et al. (2017) |
| Information Bottleneck and self-consistent equations | Tishby, Pereira & Bialek (1999) |
| IB for deep learning, information plane | Tishby & Zaslavsky (2015); Shwartz-Ziv & Tishby (2017) |
| Critique of the compression phase | Saxe et al. (2018), ICLR |
| Deep Variational Information Bottleneck | Alemi et al. (2017), ICLR |
| InfoNCE, optimal critic, $\log K$ ceiling | van den Oord, Li & Vinyals (2018); Poole et al. (2019) |
| MI neural estimation | Belghazi et al. (2018), MINE |
| Bits-back coding and MDL for weights | Hinton & van Camp (1993); Honkela & Valpola (2004); Townsend et al. (2019), *Bits-Back with ANS* |
| MDL principle | Rissanen (1978); Grünwald (2007) |
| KL-regularized RL, tilted policies, DPO | Ziegler et al. (2019); Schulman et al. (2017); Rafailov et al. (2023) |
| Scaling laws and the irreducible loss | Kaplan et al. (2020); Hoffmann et al. (2022) |
| PAC-Bayes and compression bounds | McAllester (1999); Dziugaite & Roy (2017) |